<a href="https://colab.research.google.com/github/neox-alt/data-Science/blob/main/Pruning-ViT/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import ViTModel, ViTConfig, ViTImageProcessor

from torch.utils.data import DataLoader,Dataset

from torchvision import transforms
from PIL import Image
import time
import os
import json
from collections import defaultdict


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"using the device: {device}")

In [ ]:
# getting the original model ViT base model before pruning from huggingface google/vit-base-patch16-224

from transformers import ViTForImageClassification, ViTImageProcessor

#This is the model that ew gonna use

MODEL_NAME = "google/vit-base-patch16-224"

#processor will download the image processing configurations need for the base model we are using

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)


In [ ]:
model = ViTForImageClassification.from_pretrained(MODEL_NAME, output_attentions = True , ignore_mismatched_sizes = True)

In [ ]:
model = model.to(device)
model.eval()

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch

# Use this one instead — no scripts, pure parquet
dataset = load_dataset("tanganke/resisc45", split="train[:500]")
dataset = dataset.shuffle(seed=42).select(range(500))  # ← shuffle first!


print(f"Size    : {len(dataset)} images")
print(f"Classes : {dataset.features['label'].names}")

# View samples
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
class_names = dataset.features['label'].names
for i, ax in enumerate(axes.flatten()):
    sample = dataset[i]
    ax.imshow(sample['image'])
    ax.set_title(class_names[sample['label']], fontsize=9)
    ax.axis('off')
plt.suptitle("RESISC45 — Aerial Scene Classification (500 samples)")
plt.tight_layout()
plt.show()


In [ ]:
# DataLoaders ready for ViT pipeline
class AerialDataset(Dataset):
    def __init__(self, hf_data):
        self.data = hf_data
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        s = self.data[idx]
        return self.transform(s['image'].convert('RGB')), s['label']

train_loader = DataLoader(AerialDataset(dataset.select(range(400))),
                          batch_size=16, shuffle=True)
val_loader   = DataLoader(AerialDataset(dataset.select(range(400, 500))),
                          batch_size=16, shuffle=False)

images, labels = next(iter(train_loader))
print(f"Batch shape : {images.shape}")   # [16, 3, 224, 224] ✅
print(f"Labels      : {labels.tolist()}")
print("Ready!")

In [ ]:
def compute_attention_scores_per_head(model, dataloader, num_batches =5):
  model.eval()
  head_scores = defaultdict(lambda:[])

  with torch.no_grad():
    for batch_idx,(images,_) in enumerate(dataloader):
      if batch_idx >= num_batches :
        break

      images = images.to(device)

      outputs =model(images,output_attentions =True)

      attentions = outputs.attentions

      for layer_idx , layer_attn in enumerate(attentions):
        batch_size , num_heads, seq_len,_ = layer_attn.shape

        for head_idx in range(num_heads):
          head_attn = layer_attn[:,head_idx,:,:]

          #calculating the entropy per attention head

          attn_probs = head_attn + 1e-9
          entropy = -(attn_probs * torch.log(attn_probs)).sum(dim=-1).mean()

          #Now calculating the spatial variance per head

          spatial_var = head_attn.var(dim=-1).mean()

          # --- Combined domain-aware score ---
          # Low entropy + high spatial variance = important head for small objects
          # We invert entropy so higher score = more focused = more important
          score = spatial_var - (entropy * 0.1)
          head_scores[layer_idx].append(score.item())

  averaged_scores ={}
  for layer_idx in head_scores:
    scores_per_head = np.array(head_scores[layer_idx])
    num_heads = model.config.num_attention_heads
    scores_matrix = scores_per_head.reshape(-1, num_heads)
    averaged_scores[layer_idx] = scores_matrix.mean(axis=0)
  return averaged_scores




In [ ]:
# Compute scores
head_importance = compute_attention_scores_per_head(model, val_loader, num_batches=3)

In [ ]:
print(head_importance)

In [ ]:
print("\nHead Importance Scores (higher = more important for aerial objects):")
print(f"{'Layer':<8} {'Min Score':<12} {'Max Score':<12} {'Mean Score':<12}")
print("-" * 45)
for layer_idx in sorted(head_importance.keys()):
    scores = head_importance[layer_idx]
    print(f"{layer_idx:<8} {min(scores):<12.4f} {max(scores):<12.4f} {np.mean(scores):<12.4f}")

In [ ]:
#Now we need to prune the less importance heads
def prune_head_by_Importance(model,head_importance, threshold=0.3):
  #need to prune and remove the attentions heads that has less importance
  num_layers = model.config.num_hidden_layers
  num_heads = model.config.num_attention_heads
  heads = num_layers*num_heads

  heads_to_be_pruned = int(heads * threshold)
  all_heads = []
  for layer_idx in range (num_layers):
    if layer_idx in head_importance:
      scores = head_importance[layer_idx]
      for head , score in enumerate(scores):
        all_heads.append((layer_idx, head,score))


  all_heads_sorted = sorted(all_heads, key =lambda x :x[2])

  select_heads_prune = all_heads_sorted[:heads_to_be_pruned]


  pruned_head_dict  = defaultdict(set)

  for layer_idx, head_idx, score in select_heads_prune:
    pruned_head_dict[layer_idx].add(head_idx)

  print(f"\nPruning {heads_to_be_pruned}/{heads} heads ({threshold*100:.0f}%)")
  print("\nHeads to prune per layer:")
  for layer_idx in sorted(pruned_head_dict.keys()):
      heads = sorted(pruned_head_dict[layer_idx])
      print(f"  Layer {layer_idx:2d}: heads1 {heads}")

    #count the params before pruning

  params_before = sum(p.numel() for p in model.parameters())

    # Apply pruning using HuggingFace's built-in method
    # This properly reshapes all weight matrices
    #model.prune_heads(dict(pruned_head_dict))

    #need to do the manual pruning
  for layer_idx, heads in pruned_head_dict.items():
      layer = model.vit.encoder.layer[layer_idx]
      attn  = layer.attention.attention
      head_size = model.config.hidden_size // num_heads

      for head_idx in heads:
          start = head_idx * head_size
          end   = start + head_size

          # Zero out query, key, value weights for this head
          with torch.no_grad():
              attn.query.weight.data[start:end, :] = 0
              attn.query.bias.data[start:end]      = 0
              attn.key.weight.data[start:end, :]   = 0
              attn.key.bias.data[start:end]        = 0
              attn.value.weight.data[start:end, :] = 0
              attn.value.bias.data[start:end]      = 0

    #counting the params after
  params_after = sum(p.numel() for p in model.parameters())
  reduction = (1 - params_after / params_before) * 100

  zeroed = sum(
        (p == 0).sum().item()
        for p in model.parameters()
    )

  stats = {
        "params_before": params_before,
        "Zeroed_after": zeroed,
        "reduction_pct": (zeroed / params_before) * 100,
        "heads_pruned": heads_to_be_pruned,
        "total_heads": heads
    }
  return model, dict(pruned_head_dict), stats









In [ ]:
pruned_model, pruned_heads_dict, pruning_stats = prune_head_by_Importance(
    model,
    head_importance,
    threshold=0.3   # Remove 30% of heads
)

In [ ]:

print(f"\n{'='*40}")
print(f"PRUNING RESULTS")
print(f"{'='*40}")
print(f"Parameters before: {pruning_stats['params_before']:,}")
print(f"Zeroed params:  {pruning_stats['Zeroed_after']:,}")
print(f"Reduction:         {pruning_stats['reduction_pct']:.1f}%")
print(f"Heads pruned:      {pruning_stats['heads_pruned']}/{pruning_stats['total_heads']}")

In [ ]:
#post pruning quantization
def qunatize_the_pruned_model(model):
  model_cpu = model.cpu()
  model_cpu.eval()

  quantized_model = torch.quantization.quantize_dynamic(model_cpu,{nn.Linear},dtype=torch.qint8)

  return quantized_model


In [ ]:
quntize_model = qunatize_the_pruned_model(pruned_model)

In [ ]:
#evaluate before and after

In [ ]:
def get_model_size_mb(model):
    """Calculate model size in MB."""
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.pth') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    return size_mb

In [ ]:
def latency_evaluation(model, num_runs =50, image_size=224):
  model.eval()
  dummy_input = torch.randn(1, 3, image_size, image_size)

  device = next(model.parameters()).device

  dummy_input.to(device)
  latencies = []

  with torch.no_grad():
    for _ in range(num_runs):
      start = time.perf_counter()
      _ = model(pixel_values=dummy_input)
      end = time.perf_counter()
      latencies.append((end - start) * 1000)  # Convert to ms

  return {
      "mean_ms": np.mean(latencies),
      "std_ms": np.std(latencies),
      "min_ms": np.min(latencies),
      "max_ms": np.max(latencies)
    }

In [ ]:
def count_flops(model, image_size =224):
  from fvcore.nn import FlopCountAnalysis
  dummy_input = torch.randn(1, 3, image_size, image_size)

  # Get model on CPU
  model_cpu = model.cpu() if hasattr(model, 'cpu') else model
  flops = FlopCountAnalysis(model_cpu, {"pixel_values": dummy_input})
  return flops.total() / 1e9  # Convert to GFLOPs

In [ ]:
# now we need to compare these with naseline model and the pruned/quntized model

#getting a new baseline model

baselineModel = ViTForImageClassification(MODEL_NAME,output_attention=False,ignore_mismatched_sizes=True)
baselineModel.eval()

